In [1]:
import json
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

from utils.url_features import extract_url_features, FEATURE_NAMES

In [2]:
# Load dataset
df = pd.read_csv("data/urls.csv")
df = df.dropna()
print("Dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()

Dataset shape: (411244, 2)
label
0    344800
1     66444
Name: count, dtype: int64


,url,label
0,diaryofagameaddict.com,1
1,espdesign.com.au,1
2,iamagameaddict.com,1
3,kalantzis.net,1
4,slightlyoffcenter.net,1


In [3]:
# Convert URLs into features
X = pd.DataFrame([extract_url_features(u) for u in df["url"]], columns=FEATURE_NAMES)
y = df["label"].values
X.head()

,url_length,num_dots,num_hyphens,num_digits,has_https,num_at,num_slashes,num_question,num_equals,num_underscore,has_ip,domain_length,num_subdomains,path_length,suspicious_words,digit_ratio,special_ratio
0,22,1,0,0,0,0,0,0,0,0,0,22,0,0,0,0.0,0.045455
1,16,2,0,0,0,0,0,0,0,0,0,16,1,0,0,0.0,0.125000
2,18,1,0,0,0,0,0,0,0,0,0,18,0,0,0,0.0,0.055556
3,13,1,0,0,0,0,0,0,0,0,0,13,0,0,0,0.0,0.076923
4,21,1,0,0,0,0,0,0,0,0,0,21,0,0,0,0.0,0.047619


In [4]:
# Split data (stratified, fixed seed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", len(X_train), " Test:", len(X_test))

Train: 328995  Test: 82249


In [5]:
# Train and compare models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_leaf=3,
                                            random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=20, random_state=42),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    print(f"\n{name}  accuracy: {results[name]['accuracy'] * 100:.2f}%")
    print(classification_report(y_test, pred, target_names=["Safe", "Phishing"], digits=4))


Random Forest  accuracy: 92.97%
              precision    recall  f1-score   support

        Safe     0.9372    0.9819    0.9590     68960
    Phishing     0.8750    0.6588    0.7517     13289

    accuracy                         0.9297     82249
   macro avg     0.9061    0.8203    0.8553     82249
weighted avg     0.9272    0.9297    0.9255     82249




Decision Tree  accuracy: 92.09%
              precision    recall  f1-score   support

        Safe     0.9364    0.9716    0.9537     68960
    Phishing     0.8170    0.6575    0.7286     13289

    accuracy                         0.9209     82249
   macro avg     0.8767    0.8145    0.8411     82249
weighted avg     0.9171    0.9209    0.9173     82249




Logistic Regression  accuracy: 86.42%
              precision    recall  f1-score   support

        Safe     0.8686    0.9874    0.9242     68960
    Phishing     0.7749    0.2248    0.3486     13289

    accuracy                         0.8642     82249
   macro avg     0.8217    0.6061    0.6364     82249
weighted avg     0.8535    0.8642    0.8312     82249



In [6]:
best = max(results, key=lambda n: results[n]["f1"])
print("Best URL model:", best)

joblib.dump(models[best], "models/url_model.pkl", compress=3)
json.dump({"best": best, "results": results, "train_size": len(X_train), "test_size": len(X_test),
           "feature_names": FEATURE_NAMES},
          open("models/url_metrics.json", "w"), indent=2)
print("URL model saved successfully!")

Best URL model: Random Forest


URL model saved successfully!
